# Production ML Monitoring & A/B Testing

**Chapter 10: AI Infrastructure**

## Scenario

You've deployed a **BERT sentiment classifier** to production. Now you need to:
1. **Monitor** for data drift and prediction drift
2. **Detect degradation** within 24 hours
3. **A/B test** new model versions safely
4. **Rollback** automatically if accuracy drops below threshold (in <5 min)

## Tools
- **MLflow** for model serving
- **Evidently AI** for drift detection
- **SQLite** for A/B testing metadata
- **Custom framework** for gradual rollout and automated rollback

## Constraint
Detect degradation within 24 hours, rollback in <5 minutes.

In [ ]:
# TODO: Implement this cell


### What Just Happened?

We deployed **Model v1** (BERT sentiment classifier) using MLflow:
- Registered the model in MLflow's model registry
- Created a serving endpoint (simulated at localhost:5001)
- Model predicts sentiment: 0 (negative), 1 (neutral), 2 (positive)
- Base accuracy: **85%**

In production, you'd use `mlflow models serve` or deploy to a cloud endpoint.

In [ ]:
# TODO: Implement this cell


### Production Traffic Simulation

We generated 100 synthetic production requests with:
- **Text samples**: Customer reviews/feedback
- **True labels**: Ground truth (obtained after user feedback)
- **Distribution**: Similar to training data (balanced sentiment)

Later we'll introduce **drift** to simulate real-world degradation.

In [ ]:
# TODO: Implement this cell


### Prediction Logging

Every production prediction is logged to SQLite with:
- **Timestamp**: When prediction was made
- **Predicted label**: Model output
- **True label**: Ground truth (from user feedback)
- **Model version**: v1 or v2
- **Latency**: Inference time in milliseconds
- **Correctness**: Binary flag

This enables:
1. Performance monitoring (accuracy, latency)
2. Drift detection (distribution shifts)
3. A/B test comparison (v1 vs. v2)

In [ ]:
# TODO: Implement this cell


### Evidently AI Drift Detection

**Evidently** compares training data (reference) vs. production data (current):

1. **Data Drift**: Feature distributions shifted?
   - Text length, word count, vocabulary
   
2. **Target Drift**: Label distribution changed?
   - More negative reviews than expected?

**Day 1 Results**: No drift detected (production matches training)

The HTML report includes:
- Statistical tests (KS test, chi-squared)
- Distribution plots
- Drift scores per feature

In [ ]:
# TODO: Implement this cell


### Data Drift Detection

**What happened?**
- **Day 1-7**: Production data matched training distribution
- **Day 8**: Sudden shift detected:
  - Text length decreased (shorter reviews)
  - Target distribution skewed (more negative)
  - Model accuracy dropped below 75% threshold

**Why it matters:**
Model was trained on longer, balanced reviews. New data breaks assumptions.

**Response:**
1. Alert on-call engineer
2. Investigate root cause (product issue? data pipeline bug?)
3. Deploy improved model (v2) via A/B test

In [ ]:
# TODO: Implement this cell


### Prediction Drift

**Monitoring output distribution** catches:
1. **Model degradation**: Predicting one class too often
2. **Label shift**: Real-world distribution changed
3. **Serving bugs**: Model stuck on one output

**Example:**
- Expected: 33% negative, 33% neutral, 33% positive
- Actual: 70% negative, 20% neutral, 10% positive
- **Action**: Deploy Model v2 to handle new distribution

In [ ]:
# TODO: Implement this cell


### A/B Testing Setup

**Canary deployment** (10% v2, 90% v1):
1. **Low risk**: Only 10% of users see v2
2. **Fast detection**: If v2 fails, impact is minimal
3. **Deterministic routing**: Same user always gets same model

**A/B configuration stored in SQLite:**
- Model versions + traffic percentages
- Start time and status
- Updated dynamically for gradual rollout

**Next:** Compare v1 vs. v2 metrics to decide on full rollout

In [ ]:
# TODO: Implement this cell


### A/B Test Results

**Key Metrics:**
1. **Accuracy**: Correctness rate
2. **F1 Score**: Balanced precision/recall
3. **Latency**: Response time

**Typical Results:**
- **v1**: 72% accuracy, 50ms latency (degraded due to drift)
- **v2**: 89% accuracy, 52ms latency (handles drifted data better)

**Decision:**
v2 shows **+17% accuracy** → Proceed with gradual rollout

**Statistical Significance:**
With 200 samples, a 5%+ improvement is statistically significant (p < 0.05)

In [ ]:
# TODO: Implement this cell


### Gradual Rollout Strategy

**Why gradual?**
1. **Minimize risk**: Catch issues early (at 25% traffic, not 100%)
2. **Monitor stability**: Check accuracy/latency at each step
3. **Easy rollback**: If problems arise, only some users affected

**Rollout Steps:**
- 10% → 25% → 50% → 75% → 100%
- Health check after each step
- Automatic rollback if accuracy drops

**Best Practices:**
- Monitor for at least 1 hour per step in production
- Check multiple metrics (accuracy, latency, error rate)
- Have on-call engineer ready for rollback

In [ ]:
# TODO: Implement this cell


### Automated Rollback System

**How it works:**
1. **Continuous monitoring**: Check accuracy every 60 seconds
2. **Sliding window**: Evaluate last 100 predictions
3. **Threshold trigger**: If accuracy < 75%, rollback immediately
4. **Fast execution**: Complete rollback in <5 seconds

**Rollback Steps:**
1. Detect accuracy drop
2. Update A/B config (100% traffic to v1)
3. Alert on-call engineer
4. Log incident for investigation

**Why it matters:**
- **Minimize user impact**: Bad model serves for minutes, not hours
- **Sleep better**: No need for 24/7 manual monitoring
- **Meet SLA**: <5 min rollback meets production requirements

---

## Summary

You've built a complete production ML monitoring system:

1. ✓ **Model serving** with MLflow
2. ✓ **Prediction logging** for observability
3. ✓ **Data drift detection** with Evidently AI
4. ✓ **Prediction drift monitoring** (output imbalance)
5. ✓ **A/B testing** framework (10% canary → 100%)
6. ✓ **Gradual rollout** with health checks
7. ✓ **Automated rollback** (accuracy < 75% → revert to v1)

**Key Insights:**
- Drift happens! Monitor both input (data drift) and output (prediction drift)
- A/B test new models safely (canary deployment)
- Automate rollback to meet <5 min recovery SLA
- Log everything for post-mortem analysis

**Next Steps:**
- Integrate with Prometheus/Grafana for real-time dashboards
- Add latency and error rate monitoring
- Set up PagerDuty alerts for on-call rotation
- Implement champion/challenger testing (compare multiple models)